# From linear classification to neural networks

**Lab 5 · Building and fitting a neural network**

In Labs 3 and 4, you implemented predictors and classifiers, fitted their parameters,
and examined the empirical squared loss in Lab 3 and the empirical logistic loss in Lab 4. Here you will fit and evaluate a neural-network classifier.
We supply the familiar affine classifier on a new dataset. **You will implement the
network, use automatic differentiation to obtain its gradients, adapt gradient
descent, and interpret the results using supplied plots.** Start with two hidden neurons, as
in the lecture's XOR example, then compare five fresh initialisations and keep
the run with the lowest final empirical logistic loss. Then widen the network to 100 hidden neurons.
Compare the two widths and the different initialisations on the same observations.

These experiments illustrate two **heuristic strategies from the lecture**:
trying different initialisations and changing the architecture. We explore the
second strategy by increasing the hidden width from 2 to 100 neurons. We will assess their effect using
the empirical logistic losses, mistake counts and decision boundaries.

Use **Google Colab** and work from top to bottom; no GPU is needed.
Write your code in each **YOUR CODE** cell before opening **Show answer**.
The answer gives a solution, or refers you to a solution you already wrote.
Opening an answer does not execute it: enter the code in the exercise cell and run it.
All plotting code is supplied; run those cells and interpret the figures.

**To repeat an experiment from the beginning, rerun its parameter-initialisation
cell and then its fitting cell.** Rerunning only fitting continues from the current
parameters. The stated seeds make each initialisation repeatable without changing
the existing dataset.

Use the parameter, function and result names requested in the exercises so that
the supplied checks and plots can find them. Names of local intermediate variables
are up to you. We use the functions, loops, lists and tensor operations from Labs
1–4; new operations are explained before you need them.

## 1. Generate and visualise the data

We generate **100 observations**, arranged in four Gaussian clouds. Each cloud has 25 observations, with independent coordinate noise of standard deviation $0.225$.

| Cloud centre | Class | Observations |
|---|---:|---:|
| $(0,0)$ | 0 | 25 |
| $(1,1)$ | 0 | 25 |
| $(0,1)$ | 1 | 25 |
| $(1,0)$ | 1 | 25 |

The four centres are exactly the XOR inputs from the lectures. Diagonally opposite clouds share a label. This is a noisy version of the XOR arrangement from the lectures: the inputs are points in the Euclidean plane, and each label records the class assigned to the cloud that generated the observation. Blue circles show class 0 and orange triangles show class 1, as in Lab 4.

The tensor `X` contains the two input coordinates and has shape `(N, 2)`. The labels in `y` have shape `(N, 1)`. Run the supplied cell to create and visualise the data.

In [ ]:
import torch as t
import matplotlib.pyplot as plt
t.manual_seed(1)  # Fix the random seed for this notebook.

N = 100
# Four Gaussian clouds in an XOR arrangement: 25 observations per cloud.
# spread is the standard deviation of each coordinate, not a variance.
spread = 0.225
x_class_0 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([0.0, 0.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0,  1.0])
], 0)
x_class_1 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([0.0, 1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([1.0, 0.0])
], 0)
X = t.cat([x_class_0, x_class_1], 0)              # Shape: (100, 2).
y = t.cat([t.zeros(N // 2, 1), t.ones(N // 2, 1)], 0)  # Shape: (100, 1).

# Show observed classes using the same colours and markers as in Lab 4.
plt.figure()
plt.scatter(x=x_class_0[:, 0], y=x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x=x_class_1[:, 0], y=x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.axis('equal')
plt.legend()
plt.show()

**Supplied plotting tools.** Run the next cell once. Its code is
folded away because drawing the figures is not an exercise.
Later cells call these tools with your recorded empirical logistic losses, mistake
counts and boundary scores; you only need to run those cells and
interpret the pictures. `save_boundary(net)` records the current
network's scores on a drawing grid, so we can display that stage
after the parameters have changed. This grid adds no observations
to the dataset used for fitting.

In [ ]:
#@title Supplied plotting helpers — run this cell once; no changes are needed.
import math

# This grid is only for drawing pictures; it adds no fitting observations.
x1_limits = (X[:, 0].min().item() - 0.25, X[:, 0].max().item() + 0.25)
x2_limits = (X[:, 1].min().item() - 0.25, X[:, 1].max().item() + 0.25)
x1_grid, x2_grid = t.meshgrid(t.linspace(*x1_limits, 250),
                            t.linspace(*x2_limits, 250), indexing='xy')
X_grid = t.stack([x1_grid, x2_grid], dim=-1).reshape(-1, 2)

def save_boundary(network):
    return network(X_grid).reshape(x1_grid.shape).detach()

def save_affine_boundary(theta):
    return (theta[0] + X_grid @ theta[1:]).reshape(x1_grid.shape)

def plot_affine_curves(losses, mistakes):
    updates = range(len(losses))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(updates, losses, color='0.35')
    axes[0].set(title='Affine model: empirical logistic loss', ylabel='Summed empirical logistic loss', ylim=(0, 75))
    axes[1].plot(updates, [value / math.log(2) for value in losses],
                 color='tab:purple', label=r'Empirical logistic loss / $\log 2$')
    axes[1].plot(updates, mistakes, color='tab:green', label='Mistakes')
    axes[1].set(title='Affine model\nEmpirical logistic loss and mistakes', ylabel='Value', ylim=(0, 110))
    axes[1].legend()
    for ax in axes:
        ax.set_xlabel('Completed updates')
        ax.grid(alpha=0.2)
    plt.show()

def plot_affine_boundaries(checkpoints):
    x1_line = t.tensor(x1_limits)
    fig, ax = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
    ax.scatter(x_class_0[:, 0], x_class_0[:, 1], color='tab:blue', label='Class 0')
    ax.scatter(x_class_1[:, 0], x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
    for (step, theta), style in zip(checkpoints.items(), (':', '--', '-.', '-')):
        b, w1, w2 = theta[:, 0]
        x2_line = -(b + w1 * x1_line) / w2
        ax.plot(x1_line, x2_line, color='black', linestyle=style,
                label=f'After {step} update' + ('s' if step != 1 else ''))
    ax.set(xlim=x1_limits, ylim=x2_limits, xlabel='$x_1$', ylabel='$x_2$',
           title='Affine decision boundaries')
    ax.set_aspect('equal')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.show()

def plot_first_comparison(affine_losses, affine_mistakes, network_losses, network_mistakes):
    updates = range(len(network_losses))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    axes[0].plot(updates, affine_losses, '--', color='0.4', label='Affine score')
    axes[0].plot(updates, network_losses, color='tab:purple', label='Two-neuron network')
    axes[0].set(title='Empirical logistic loss', ylabel='Summed empirical logistic loss')
    for ax, losses, mistakes, title in zip(
            axes[1:], [affine_losses, network_losses], [affine_mistakes, network_mistakes],
            ['Affine model\nEmpirical logistic loss and mistakes', 'Two-neuron network\nEmpirical logistic loss and mistakes']):
        ax.plot(updates, t.tensor(losses) / math.log(2), color='tab:purple', label=r'Empirical logistic loss / $\log 2$')
        ax.plot(updates, mistakes, color='tab:green', label='Mistakes')
        ax.set(title=title, ylabel='Value', ylim=(0, 1.1 * max(100, max(losses) / math.log(2))))
    for ax in axes:
        ax.set_xlabel('Completed updates')
        ax.grid(alpha=0.2)
        ax.legend()
    plt.show()

def plot_network_curves(labels, loss_histories, error_histories):
    assert len(labels) == len(loss_histories) == len(error_histories)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    styles = ['--', '-'] if len(labels) == 2 else [':', '--', '-', '-.', (0, (5, 1, 1, 1, 1, 1)), (0, (7, 3))]
    for index, (label, losses, mistakes) in enumerate(zip(labels, loss_histories, error_histories)):
        updates = range(len(losses))
        style = styles[index % len(styles)]
        axes[0].plot(updates, losses, linestyle=style, color='tab:purple', label=label)
        axes[1].plot(updates, mistakes, linestyle=style, color='tab:green', label=label)
    axes[0].set(title='Empirical logistic loss', ylabel='Summed empirical logistic loss')
    axes[1].set(title='Classification mistakes', ylabel='Mistakes')
    for ax in axes:
        ax.set_xlabel('Completed updates')
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8 if len(labels) > 3 else 10)
    plt.show()

def draw_boundary(ax, score_grid, title):
    classes = (score_grid > 0).to(t.float32)
    ax.contourf(x1_grid, x2_grid, classes,
                levels=[-0.5, 0.5, 1.5], colors=['#edf3fb', '#fff1e9'])
    ax.contour(x1_grid, x2_grid, score_grid, levels=[0], colors='black', linewidths=1)
    class_0 = y[:, 0] == 0
    class_1 = y[:, 0] == 1
    ax.scatter(X[class_0, 0], X[class_0, 1], color='tab:blue', s=18, label='Class 0')
    ax.scatter(X[class_1, 0], X[class_1, 1], color='tab:orange', marker='^', s=20, label='Class 1')
    ax.set(xlim=x1_limits, ylim=x2_limits, xlabel='$x_1$', ylabel='$x_2$', title=title)
    ax.set_aspect('equal')

def plot_boundaries(score_grids, labels, mistake_counts):
    assert len(score_grids) == len(labels) == len(mistake_counts)
    width = {2: 10, 3: 13, 4: 15}[len(score_grids)]
    fig, axes = plt.subplots(1, len(score_grids), figsize=(width, 4), constrained_layout=True)
    for ax, scores, label, mistakes in zip(axes, score_grids, labels, mistake_counts):
        draw_boundary(ax, scores, f'{label}\nMistakes: {mistakes}')
    axes[0].legend(loc='upper left', fontsize=8)
    plt.show()

def plot_lecture_comparison(original_scores, original_loss, original_mistakes, lecture_network):
    scores = lecture_network(X)
    lecture_loss = loss_from_scores(scores).item()
    lecture_mistakes = mistakes_from_scores(scores)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
    draw_boundary(axes[0], original_scores,
                  f'Original fitted two-neuron network\nEmpirical logistic loss: {original_loss:.3f}\nMistakes: {original_mistakes}')
    draw_boundary(axes[1], save_boundary(lecture_network),
                  f'Two-neuron network from the lecture\nEmpirical logistic loss: {lecture_loss:.3f}\nMistakes: {lecture_mistakes}')
    axes[0].legend(loc='upper left', fontsize=8)
    plt.show()

def plot_restart_curves(seeds, losses, mistakes, original_losses, original_mistakes, selected_seed):
    labels = ['Original initialisation'] + [
        f'Seed {seed}' + (' (selected)' if seed == selected_seed else '')
        for seed in seeds]
    plot_network_curves(labels, [original_losses] + [losses[seed] for seed in seeds],
                        [original_mistakes] + [mistakes[seed] for seed in seeds])

def plot_restart_boundaries(seeds, grid_scores, losses, mistakes,
                            original_scores, original_loss, original_mistakes, selected_seed):
    assert len(seeds) == 5
    fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
    panels = axes.ravel()
    draw_boundary(panels[0], original_scores,
                  f'Original initialisation\nEmpirical logistic loss: {original_loss:.3f}\nMistakes: {original_mistakes}')
    for ax, seed in zip(panels[1:], seeds):
        label = f'Seed {seed}' + (' — selected' if seed == selected_seed else '')
        draw_boundary(ax, grid_scores[seed],
                      f'{label}\nEmpirical logistic loss: {losses[seed][-1]:.3f}\nMistakes: {mistakes[seed][-1]}')
    panels[0].legend(loc='upper left', fontsize=8)
    plt.show()

## 2. Reuse the linear classifier from Lab 4

Look at the four clouds. **Could a single straight decision boundary separate the four centres according to their labels?** Keep your prediction in mind while running the model.

The code below supplies the completed calculations from Lab 4: the stable empirical logistic loss, its gradient and the number of classification mistakes. You do not need to implement them again. Recall why we use `logsigmoid` to evaluate the logarithms directly.

As in Lab 4, we include the intercept using a column of ones. Here that design matrix is called `X_affine`, while `X` keeps the original coordinates for the later network. The functions are prefixed with `affine_` to keep the two calculations separate. Their mathematical definitions are unchanged.

The coefficient column contains $(b,w_1,w_2)$, and the scores are `X_affine @ theta`. The predicted label is 1 when the sigmoid of the score is greater than 0.5, and 0 otherwise, including at equality. This is equivalent to testing whether the score is strictly positive.

In [ ]:
# Supplied: the completed functions from Lab 4, with affine-specific names.
X_affine = t.cat([t.ones(N, 1), X], dim=1)  # Shape (N, 3), including the intercept.

def affine_loss(theta):
    scores = X_affine @ theta
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

def affine_grad(theta):
    scores = X_affine @ theta
    probabilities = t.sigmoid(scores)
    return X_affine.mT @ (probabilities - y)

def affine_mistakes(theta):
    probabilities = t.sigmoid(X_affine @ theta)
    predicted_labels = (probabilities > 0.5).to(y.dtype)
    return int((predicted_labels != y).sum().item())

### 2.1. Run gradient descent

Run the supplied update loop. We use Lab 4's zero initialisation and learning rate `0.01`, now allowing **1,000 updates**. Every update uses the full dataset.

As before, record the initial empirical logistic loss and mistake count, then record them again **after** each update. Entry `step` in each history therefore describes exactly `step` completed updates. We also store the parameter tensors at selected steps for the boundary plot.


**Recording results.** A Python list such as `[first_value]` starts with one entry;
`history.append(value)` adds another at the end. `history[0]` selects the first
entry, and `history[-1]` the last. Here entry `step` records that many completed
updates. `range(1, num_updates + 1)` includes 1 through `num_updates`, because the
upper endpoint of `range` is excluded. `.item()` extracts an ordinary Python
number from a tensor containing one value.

`affine_checkpoints = {}` creates an empty **dictionary**, which stores values
under keys. `affine_checkpoints[step] = affine_theta` stores the current parameter
tensor under the update number `step`.
The condition `step in (...)` checks whether `step` is one of the listed numbers.

The printed table uses **f-strings**: in `f'Update {step}'`, Python inserts the
value of `step` into the text. A format such as `{value:.3f}` displays three
decimal places; table-width and precision settings only affect the display.

In [ ]:
affine_theta = t.zeros(3, 1)
affine_learning_rate = 0.01
num_updates = 1000
affine_loss_history = [affine_loss(affine_theta).item()]
affine_error_history = [affine_mistakes(affine_theta)]
affine_checkpoints = {}

for step in range(1, num_updates + 1):
    affine_theta = affine_theta - affine_learning_rate * affine_grad(affine_theta)
    affine_loss_history.append(affine_loss(affine_theta).item())
    affine_error_history.append(affine_mistakes(affine_theta))

    # Supplied: save this tensor; each later update creates a new one.
    if step in (1, 10, 100, num_updates):
        affine_checkpoints[step] = affine_theta

print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, num_updates):
    print(f'{step:7d}   {affine_loss_history[step]:23.6f}   {affine_error_history[step]:8d}')

### 2.2. Inspect the curves and the boundaries

Run the supplied plotting cells. These are the same quantities we examined in Lab 4:

- the summed empirical logistic loss;
- the empirical logistic loss divided by $\log 2$, alongside the number of mistakes;
- the decision boundaries at successive updates.

The division by $\log 2$ is only for display: **the updates use the original, unscaled empirical logistic loss**. Each mistake contributes at least $\log 2$ to that empirical logistic loss, so the scaled empirical logistic loss bounds the mistake count from above.

All affine boundaries have the form $b+w_1x_1+w_2x_2=0$. For these saved coefficients, $w_2$ is nonzero, so we can draw the lines using $x_2=-(b+w_1x_1)/w_2$, just as in Lab 4. This plotting code is supplied; focus on interpreting its output.

In [ ]:
# Supplied: the affine empirical logistic loss and mistake curves.
plot_affine_curves(affine_loss_history, affine_error_history)

In [ ]:
# Supplied: draw the saved affine boundaries and keep its final picture.
plot_affine_boundaries(affine_checkpoints)
affine_grid_scores = save_affine_boundary(affine_theta)

### Pause and interpret: what has gradient descent achieved?

1. Compare the initial and final empirical logistic losses and mistake counts. Has the affine model fitted the observations well? The empirical logistic loss curve looks almost flat: does this mean that no updates have taken place? Use the printed values to read changes that are too small to see in the curves.
2. What happens to the boundary between updates 1, 100 and 1,000? Could more updates turn it into a boundary with several segments?
3. Why do the empirical logistic loss and the number of mistakes not necessarily decrease together?

Write down your answers before opening the explanation.

<details>
<summary>Show answer — interpreting the affine fit</summary>

**1. The empirical logistic loss decreases very little, but the mistake count first falls and then rises.** With the supplied seed, the empirical logistic loss changes from approximately $69.3147$ to $69.3083$. The mistake count starts at 50, falls to 33 after 3 updates (and remains at 33 after 4), then rises to 47 by update 100. The empirical logistic loss has already essentially stopped changing by update 100. The final model still misclassifies almost half of these observations.

The empirical logistic loss curve looks flat because the total decrease is only about $0.00645$. Initially, all coefficients are zero, so every score is zero and every predicted probability is $1/2$. The summed empirical logistic loss is therefore $N\log 2\approx69.3147$. The purple curve in the right panel shows this empirical logistic loss divided by $\log 2$, so it starts at 100 and remains close to 100.

For this nearly symmetric XOR sample, the fitted affine scores remain close to zero: a single straight decision boundary cannot capture the main XOR pattern. Small sample asymmetries allow a slight reduction in the empirical logistic loss, but the probabilities remain near $1/2$. Small changes in these probabilities can move observations across the classification threshold and change their predicted labels. The empirical logistic loss is convex in the affine coefficients, so this behaviour is not due to a suboptimal local minimum. It illustrates a limitation of the affine model on these data.

**2. Updates change the coefficients of a straight line.** The boundary initially rotates and shifts; the lines at 100 and 1,000 updates almost coincide. Further updates can change its slope and intercept, but an affine score always has a straight boundary when its feature weights are nonzero. A single straight line cannot separate the four cloud centres according to their XOR labels. We need a richer family of scores to express a different shape.

**3. Gradient descent reduces the empirical logistic loss, not directly the number of mistakes.** The empirical logistic loss depends on the probabilities assigned to the observed labels. The mistake count only checks which side of the classification threshold each observation falls on. Probabilities can change without changing any labels, and a decrease in total empirical logistic loss can also accompany an increase in mistakes.

We see this directly here: between updates 4 and 100, the empirical logistic loss falls from approximately $69.3121$ to $69.3083$, while the mistake count rises from 33 to 47. Crossing the threshold changes an observation's contribution to the mistake count abruptly, but changes its logistic loss continuously. Gradient descent follows the derivatives of the total empirical logistic loss; it does not choose each update to minimise the mistake count.

The final 47 mistakes are therefore the outcome of fitting the empirical logistic loss in this run, not the smallest mistake count attainable by a straight boundary. Indeed, the earlier model already made only 33 mistakes, although its empirical logistic loss was slightly larger.

</details>

## 3. Can we do better with a neural network?

Keep the **same observations, labels and empirical logistic loss**, and change the
model that produces the scores. In the lectures, the hidden weights and biases
made the feature functions adjustable. You will now implement that model and fit
all of its parameters.

In Lab 4, we wrote the gradient formula ourselves. Here, PyTorch's **automatic
differentiation** will calculate the derivatives for us. You will still implement
the gradient-descent updates. The next lecture explains **backpropagation**, the
algorithm behind this derivative calculation.

### Reuse the empirical logistic loss

Our Lab 4 empirical logistic loss function calculated the affine scores internally. The supplied
`loss_from_scores` instead takes already-computed scores and evaluates the same
empirical logistic loss, using the labels in `y`:

$$\mathcal L_{\mathrm{log}} = \sum_{i=1}^N
\ell_{\mathrm{log}}(\sigma(s_i),y_i).$$

We can use it with either affine or neural-network scores. Recall Lab 4's numerical
stability issue: we again use `logsigmoid` to evaluate the logarithms directly.
**Pass scores to this function without first applying a sigmoid.** You do not need
to implement this familiar empirical logistic loss again.

In [ ]:
def loss_from_scores(scores):
    # scores contains classification scores, not probabilities or weights.
    # Evaluate sum_i ell_log(sigmoid(score_i), y_i), using the fixed labels y.
    # logsigmoid computes the logarithms directly, without forming probabilities.
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

### 3.1. From the architecture to parameter tensors

Start with **two input coordinates, two hidden neurons and one scalar output**,
matching the architecture of the lecture's XOR example. In the notes' notation,
$L=2$, $d_0=2$, $d_1=2$ and $d_2=1$. Here $N=100$ counts observations;
$d_1$ counts hidden neurons and will change later. For one input row,

$$
\begin{aligned}
\boldsymbol a_1(\boldsymbol x)&=\boldsymbol x\boldsymbol W_1+\boldsymbol b_1,\\
\boldsymbol\phi_1(\boldsymbol x)&=\operatorname{\mathbf{ReLU}}(\boldsymbol a_1(\boldsymbol x)),\\
f_{\boldsymbol\theta}(\boldsymbol x)&=\boldsymbol\phi_1(\boldsymbol x)\boldsymbol W_2+\boldsymbol b_2.
\end{aligned}
$$

The subscript identifies the layer. The column $\boldsymbol\theta$ collects all
weights and biases mathematically; our code will store them in four tensors.
`X` has shape `(N, 2)` and contains only the original coordinates. Unlike
`X_affine`, it needs no extra column of ones: biases are stored separately.

`t.randn(rows, columns, requires_grad=True)` creates a matrix of random initial
values and asks PyTorch to track computations involving it. This will let us
differentiate an empirical logistic loss with respect to its entries later; it does not compute any
derivatives yet. With one size argument, `t.randn(size, requires_grad=True)`
creates a one-dimensional tensor. We use one-dimensional tensors for the biases:
adding a bias vector to a matrix adds it to each row (**broadcasting**).

Recall that `A.shape` gives a tensor's dimensions. You can count its scalar
entries from those dimensions, or use `A.numel()` to obtain that count directly.
Use separate `print(...)` statements if helpful; no printing loop is required.

**Exercise 1 — initialise the parameters.**

1. Work out the shapes of `W1`, `b1`, `W2` and `b2` from the matrix products above.
2. Create them using `t.randn(..., requires_grad=True)`. Use the dimension variables
   supplied below, and create them in the order **`b1`, `W1`, `b2`, `W2`** so that
   your random draws match the reference experiment.
3. Print their shapes. How many adjustable scalar parameters are there altogether?

The supplied `t.manual_seed(94)` fixes the sequence used to draw the
initial parameters. Running this cell again recreates the same starting values.
It does not change the already-created data `X` and labels `y`.

In [ ]:
t.manual_seed(94)  # Supplied: repeatable starting parameters.
input_features = 2
hidden_features = 2
output_features = 1

# YOUR CODE: create b1, W1, b2 and W2; print shapes and count scalar parameters.

<details>
<summary>Show answer — parameter shapes and initialisation</summary>

The shapes are `W1`: `(2, 2)`, `b1`: `(2,)`, `W2`: `(2, 1)`, and `b2`: `(1,)`.
There are $2\times2+2+2\times1+1=9$ scalar parameters. The notes' bias rows are
represented by one-dimensional tensors here.

```python
t.manual_seed(94)  # Supplied: repeatable starting parameters.
input_features = 2
hidden_features = 2
output_features = 1

b1 = t.randn(hidden_features, requires_grad=True)
W1 = t.randn(input_features, hidden_features, requires_grad=True)
b2 = t.randn(output_features, requires_grad=True)
W2 = t.randn(hidden_features, output_features, requires_grad=True)
print('W1:', W1.shape, 'b1:', b1.shape)
print('W2:', W2.shape, 'b2:', b2.shape)
print('Scalar parameters:', W1.numel() + b1.numel() + W2.numel() + b2.numel())
```

</details>

### 3.2. Implement the network calculation

The equations above describe one observation. Applying them to the whole matrix
`X` performs the same calculation for every row, using the same parameters.
`t.relu(A)` applies $\max\{0,a\}$ separately to every entry of `A`; it implements
the componentwise ReLU from the notes.

**Exercise 2 — write `net(X)`.** Translate the three equations into a Python
function returning the scores. You can use intermediate variables such as `a1`,
`phi1` and `scores`; these local names do not affect the supplied checks or plots.
Use the four tensors you have just created; do not draw new parameters inside
the function. Before running it, predict the shapes of all three intermediate
results for the dataset. What should happen if you pass only one input row as
`X[:1]`? This slice keeps a matrix with shape `(1, 2)`; `X[0]` would instead
return a one-dimensional tensor.

Use `@` for matrix multiplication and keep these calculations as PyTorch tensors.
The function must return its scores, not just print them. We will differentiate
this calculation later, so do not use `.item()` or `detach()` inside `net`.

The supplied checks after your implementation use `assert condition`: if the
condition is false, execution stops with an `AssertionError`. These checks test
the output shapes; they do not establish that every part of an implementation
is correct. If a check fails, inspect the intermediate shapes and formulas.

The hidden activations are values of learned feature functions. The final output
is a classification **score**, not a probability or a predicted label.

In [ ]:
# YOUR CODE: define net(X).

<details>
<summary>Show answer — the network calculation</summary>

`a1` and `phi1` have shape `(N, 2)`; `scores` has shape `(N, 1)`.
For a single-row input, `a1` and `phi1` have shape `(1, 2)`, and `scores` has
shape `(1, 1)`. The variables contain values for the inputs passed to the
function; the parameter tensors are reused.

```python
def net(X):
    a1 = X @ W1 + b1
    phi1 = t.relu(a1)
    scores = phi1 @ W2 + b2
    return scores
```

</details>

In [ ]:
# Supplied checks: one score per observation, also for a single-row input.
assert net(X).shape == y.shape
assert net(X[:1]).shape == (1, 1)
print('Dataset output shape:', net(X).shape)

### 3.3. Evaluate the empirical logistic loss and mistakes

Recall the operations used to count mistakes in Lab 4:

- Comparisons such as `A > 0` and `A != B` act entry by entry and return tensors
  of `True`/`False` values. `!=` means "not equal".
- Summing a Boolean tensor with `.sum()` counts its `True` entries.
- `A.dtype` describes the type of its entries; `B.to(A.dtype)` converts `B` to
  that type. This lets Boolean predictions be represented numerically as 0 and 1.
- `.item()` extracts a Python number from a one-value tensor, and `int(...)`
  converts a number to a Python integer.

Keep predicted labels and observed labels at the same shape `(N, 1)` when you
compare them. Work out which comparison identifies an incorrect prediction.

**Exercise 3 — evaluate your initial model.**

1. Calculate its scores and empirical logistic loss using `loss_from_scores`.
2. Write `mistakes_from_scores(scores)`, returning the number of incorrect labels
   as a Python integer. Like the empirical logistic loss helper, it should use the fixed labels `y`.
   Reuse Lab 4's decision rule: class 1 for a strictly positive score and class 0
   otherwise, including when the score is zero.
3. Print the empirical logistic loss and the number of mistakes before fitting.

Do not add a sigmoid to the network output for the empirical logistic loss.
To repeat this initial evaluation after fitting, rerun Exercise 1 first.

In [ ]:
# YOUR CODE: define mistakes_from_scores, then evaluate the initial model.

<details>
<summary>Show answer — initial empirical logistic loss and mistakes</summary>

Compare each predicted label with its observed label and count disagreements.
The supplied initialisation gives empirical logistic loss about $130.990$ and 50 mistakes. These are
the randomly initialised model's results before any fitting updates.

```python
def mistakes_from_scores(scores):
    predicted_labels = (scores > 0).to(y.dtype)
    return int((predicted_labels != y).sum().item())

scores = net(X)
L = loss_from_scores(scores)
print('Initial empirical logistic loss:', L.item())
print('Initial mistakes:', mistakes_from_scores(scores))
```

</details>

In [ ]:
# Supplied checks on artificial scores, including ties at zero.
assert mistakes_from_scores(t.zeros_like(y)) == 50
assert mistakes_from_scores(2 * y - 1) == 0
print('Mistake-count checks passed.')

### 3.4. Ask PyTorch for the derivatives

Here is a small example separate from the network. Predict the two partial
derivatives of $Q(u)=u_0^2+u_1^2$ at $u=(1,-2)$, then run the cell.
`Q.backward()` computes the derivatives and stores them in `u.grad`.

In [ ]:
u = t.tensor([1.0, -2.0], requires_grad=True)
Q = (u ** 2).sum()
Q.backward()
print('u:', u)
print('Gradient of Q:', u.grad)

The gradient is $(2,-4)$ and `u` has not changed. **`L.backward()` computes
derivatives of the scalar loss; it does not update the parameters.**

For our network, PyTorch saves these derivatives in `W1.grad`, `b1.grad`,
`W2.grad` and `b2.grad`. Each gradient has the same shape as its parameter:
`W1.grad[0, 0]` is the derivative with respect to `W1[0, 0]`. These parameter
tensors were created directly with `requires_grad=True` (they are *leaf tensors*).
Intermediate activations and scores do not normally retain their own `.grad`.

**Gradients accumulate.** If derivatives are already stored, `backward()` adds
to them. Computing a new loss does not clear them. Accumulation lets us add the
gradients of losses calculated separately; here our empirical logistic loss already sums
over all observations, so we need just one derivative calculation per update.

Before each derivative calculation, we therefore set each parameter's `.grad`
to `None`, Python's value for “nothing stored”. This removes old gradients without
changing the parameters; the next `backward()` stores fresh gradients. The loop
below does this once for each of the four parameter tensors. It also works when
no gradients have been calculated yet.

PyTorch uses 0 as the ReLU multiplier at input 0. This is a computational
convention; the ordinary derivative remains undefined there.

**Exercise 4 — compute the network's gradients.** After the supplied clearing
code, calculate a fresh empirical logistic loss, call `L.backward()`, and print each
parameter's shape and its gradient's shape. For example, compare `W1.shape`
and `W1.grad.shape`.

Explain why the shapes match. If one entry of `W1.grad` is positive, should the
corresponding weight increase or decrease in gradient descent with a positive
learning rate? These derivatives refer to the current parameters; rerun
Exercise 1 first if you want to examine the initial parameters again.

In [ ]:
# Supplied: clear previously stored gradients.
for parameter in (b1, W1, b2, W2):
    parameter.grad = None

# YOUR CODE: calculate the empirical logistic loss, differentiate it, and print the shapes.

<details>
<summary>Show answer — derivatives of the network empirical logistic loss</summary>

Each derivative corresponds to one parameter entry, so the shapes agree.
A positive derivative gives a negative change in that weight: gradient descent
subtracts the learning rate times the derivative.

```python
for parameter in (b1, W1, b2, W2):
    parameter.grad = None

L = loss_from_scores(net(X))
L.backward()
print('W1:', W1.shape, W1.grad.shape)
print('b1:', b1.shape, b1.grad.shape)
print('W2:', W2.shape, W2.grad.shape)
print('b2:', b2.shape, b2.grad.shape)
```

</details>

### 3.5. Implement gradient descent

You already know the update rule: subtract the learning rate times each current
partial derivative from the corresponding parameter.

**Using `with t.no_grad():` is the standard way to perform these manual
gradient-descent updates in PyTorch.** PyTorch has an internal **gradient-tracking
switch**. When it is on, operations involving our parameters with
`requires_grad=True` are recorded so that `backward()` can calculate derivatives.
When it is off, numerical calculations still run, but their operations are not
recorded for differentiation.

Here `t` is our name for PyTorch, and `t.no_grad()` creates an object that manages
this switch. **`with` is a Python keyword** that uses this object to:

1. Save the current setting and switch tracking off on entering the block.
2. Execute the indented lines with tracking off.
3. Restore the previous setting on leaving the block, even if an error occurs.

First calculate the empirical logistic loss and call `backward()` **outside** this block. Inside,
use the stored gradients to update the parameters. Only the tracking setting
is temporary: the parameter updates remain after the block. The gradients
already stored in `.grad` remain available, and the parameters still have
`requires_grad=True`. In our loop, tracking is back on for the next empirical logistic loss
calculation using the updated parameters.

Without this block, our first in-place parameter update with `.add_(...)` would
raise an error and stop the cell before changing that parameter. Calling
`backward()` does not itself switch tracking off.

**`.add_(...)` changes a tensor in place.** The following cell applies one update
to the separate toy variable `u` above. Then `u.grad = None` clears the stored
derivatives without changing `u`. Run the toy derivative cell again first if
you want to repeat this example from its initial values.

For our parameter tensors, use these in-place updates rather than assigning a
new tensor to each parameter name. This keeps the same parameter tensors, still
requiring gradients, available for the next derivative calculation.

In [ ]:
with t.no_grad():
    u.add_(-0.1 * u.grad)
u.grad = None
print('After the toy update:', u)  # (0.8, -1.6)
print('Cleared gradient:', u.grad)  # None

**Exercise 5 — adapt gradient descent to all four parameter tensors.**

Use 1,000 full-dataset updates with learning rate `0.001`. We keep these settings
for every neural network. The affine model uses a different learning rate because
its parameterisation differs; we compare the resulting fits, not iteration speed.

**To start again, rerun Exercise 1 before this cell.** Running only this cell
continues fitting the current parameters.

1. Create `loss_history` and `error_history` as new lists containing the initial
   empirical logistic loss and mistake count. Use `.item()` to store an empirical logistic loss as a Python number;
   `mistakes_from_scores` already returns an integer.
2. At the start of each iteration, set all four `.grad` attributes to `None`,
   as in Exercise 4. Compute a fresh empirical logistic loss and call `backward()` once.
3. Inside the supplied `no_grad` block, update all four parameter tensors.
4. Recompute the scores after updating and append the new empirical logistic loss and mistake count
   to the histories. All derivatives must be computed before any parameter changes.

Entry `step` must describe `step` completed updates: both histories will have
1,001 entries. Use these names for the supplied plots. If your lists have other
names, assign them to `loss_history` and `error_history` at the end of the cell.

Keep the supplied recording lines. `save_boundary(net)` evaluates the current
network on the plotting grid and saves values for a picture; it does not change
the parameters or the empirical logistic loss. `boundary_scores[step]` stores that picture
under the completed update number, just like the affine checkpoints.

In [ ]:
learning_rate = 0.001
num_updates = 1000
boundary_scores = {}

# YOUR CODE: create loss_history and error_history, including update 0.

for step in range(1, num_updates + 1):
    # YOUR CODE: clear gradients, calculate a fresh empirical logistic loss and differentiate it.

    with t.no_grad():
        # YOUR CODE: update every parameter using its gradient.

    # YOUR CODE: recompute scores and append empirical logistic loss and mistakes to the histories.

    # Supplied: record the boundary at these completed updates.
    if step in (1, 100, num_updates):
        boundary_scores[step] = save_boundary(net)

<details>
<summary>Show answer — the gradient-descent loop</summary>

The update rule is the same as in Lab 4. `backward()` now calculates the
derivatives for all four tensors. We clear gradients before calculating them,
update the parameters, then measure the new empirical logistic loss and mistakes.

```python
learning_rate = 0.001
num_updates = 1000
boundary_scores = {}

scores = net(X)
loss_history = [loss_from_scores(scores).item()]
error_history = [mistakes_from_scores(scores)]

for step in range(1, num_updates + 1):
    for parameter in (b1, W1, b2, W2):
        parameter.grad = None
    L = loss_from_scores(net(X))
    L.backward()

    with t.no_grad():
        b1.add_(-learning_rate * b1.grad)
        W1.add_(-learning_rate * W1.grad)
        b2.add_(-learning_rate * b2.grad)
        W2.add_(-learning_rate * W2.grad)

    scores = net(X)
    loss_history.append(loss_from_scores(scores).item())
    error_history.append(mistakes_from_scores(scores))

    # Supplied: record the boundary at these completed updates.
    if step in (1, 100, num_updates):
        boundary_scores[step] = save_boundary(net)
```

</details>

In [ ]:
# Supplied: check recording and display progress.
assert len(loss_history) == len(error_history) == num_updates + 1
print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, num_updates):
    print(step, loss_history[step], error_history[step])

Run this supplied cell after the original two-neuron fit. It keeps the results
under names beginning with `narrow_` for later comparisons. Subsequent fitting
cells create new lists and dictionaries, leaving these saved results available.
To reproduce this experiment later, rerun Exercise 1, Exercise 5 and this cell.

In [ ]:
# Supplied: keep the original two-neuron results for comparisons.
assert W1.shape == (2, 2), 'Run the original two-neuron initialisation and fit first.'
narrow_loss_history = loss_history
narrow_error_history = error_history
narrow_boundary_scores = boundary_scores
narrow_grid_scores = boundary_scores[num_updates]

## 4. Examine what your network learned

### 4.1. Inspect the empirical logistic loss and mistakes

**Exercise 6 — interpret the empirical logistic loss and mistake curves.** Run the complete supplied
cell below. It uses the original two-neuron results preserved by the preceding
supplied cell in `narrow_loss_history` and `narrow_error_history`:

- Left: empirical logistic loss against completed updates, for both models.
- Middle: affine empirical logistic loss divided by $\log 2$, alongside its mistake count.
- Right: the same two quantities for your neural network.

How much does the network's empirical logistic loss decrease? Does its mistake count decrease in
the same way? Compare the final fit with the affine model.

The scaling is for display only: each mistake contributes at least $\log 2$ to
the summed empirical logistic loss, so mistakes cannot exceed empirical logistic loss divided by $\log 2$.
**The fitting updates used the unscaled empirical logistic loss.**

In [ ]:
# Supplied: compare the affine classifier with the original two-neuron fit.
plot_first_comparison(affine_loss_history, affine_error_history,
                      narrow_loss_history, narrow_error_history)

### 4.2. Inspect the decision boundaries

A neural network's zero-score set need not be a single straight line.
Our supplied plots use scores on a grid to draw this set. The black
line separates light blue class-0 regions from light orange class-1
regions. At exactly zero, our classifier selects class 0.
The grid is only for drawing: the empirical logistic losses and mistakes still use the
same 100 observations.

**Exercise 7 — interpret the decision boundaries.**

Run the complete supplied cell below. It shows the affine fit and the original
two-neuron network after 1, 100 and 1,000 updates. The supplied saving cell kept
those three stages in `narrow_boundary_scores`; `narrow_error_history` supplies
the mistake counts. These remain the original results when we fit other networks
later. The counts refer to the observed dataset; the grid only draws the picture.

How does the boundary change during fitting? Does its final shape separate the
four clouds successfully?

In [ ]:
# Supplied: the affine fit and three stages of the original two-neuron fit.
plot_boundaries(
    [affine_grid_scores, narrow_boundary_scores[1],
     narrow_boundary_scores[100], narrow_boundary_scores[1000]],
    ['Affine: 1000 updates', '2-neuron NN: 1 update',
     '2-neuron NN: 100 updates', '2-neuron NN: 1000 updates'],
    [affine_error_history[-1], narrow_error_history[1],
     narrow_error_history[100], narrow_error_history[1000]])

## 5. Can we find a better two-neuron network?

We have fitted a neural network, yet its final empirical logistic loss (about
69.308) and 47 mistakes are almost the same as the affine
classifier's. Its boundary also looks much like a straight line. What went wrong?

In the lecture, this same architecture could represent a boundary that correctly
classified all four XOR inputs. Our observations now have noisy coordinates,
but how would that particular network perform on them? Let us implement the
lecture's score and compare its empirical logistic loss, mistakes and boundary with our fitted network.

### 5.1. Try the network constructed in the lecture

Our cloud centres are the lecture's four Boolean inputs. Use the same score:

$$s(\boldsymbol x)=\operatorname{ReLU}(x_1-x_2)
+\operatorname{ReLU}(x_2-x_1)-\frac12=|x_1-x_2|-\frac12.$$

It is positive at the class-1 centres $(0,1)$ and $(1,0)$, and negative at the
class-0 centres $(0,0)$ and $(1,1)$.

**Exercise 8 — compare the ideal inputs with the noisy observations.**

1. Write `lecture_score(inputs)` for this score, returning one column of scores.
   Use slices such as `inputs[:, 0:1]` to preserve the column dimension.
2. Count its mistakes on the four supplied centres, using `centre_y`. Our helper
   `mistakes_from_scores` uses the 100 labels in `y`, so do not use it for the centres.
3. Evaluate the same score on the observed data `X`; now use `mistakes_from_scores`
   and `loss_from_scores` to report its mistakes and empirical logistic loss.
4. Explain why the lecture's exact classification of four inputs no longer
   guarantees correct labels for every observation in the clouds. Do your results
   show that this finite sample cannot be classified perfectly by any network?
5. Compare this empirical logistic loss with `narrow_loss_history[-1]`, the empirical logistic loss of your
   fitted two-neuron network. Both scores use the same architecture: can the
   parameters found by gradient descent be a global minimiser of its empirical logistic loss?

Use this separate function without changing `net` or its learned parameters.

In [ ]:
centre_X = t.tensor([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
centre_y = t.tensor([[0.0], [0.0], [1.0], [1.0]])

# YOUR CODE: define lecture_score and compare the centres with the noisy data.

<details>
<summary>Show answer — ideal XOR versus noisy observations</summary>

The constructed score makes **0 mistakes on the centres**, but **8 on the noisy
observations**, with empirical logistic loss about $54.071$. Each observation keeps its
cloud's label while its coordinates are perturbed. Some observations move across
the boundary that correctly separates the four centres. The construction therefore
does not inherit a guarantee of zero mistakes on these new inputs.

This shows the effect of the noisy coordinates on this particular classifier.
It does **not** prove that no classifier can fit all 100 observed labels. Nor can
we attribute every error of our fitted network solely to noise: its result also
depends on the available functions, initialisation and optimisation.

The constructed score uses two hidden neurons, just like our fitted network,
but improves both its empirical logistic loss ($54.071$ instead of $69.308$) and its
mistake count (8 instead of 47). We therefore know that the fitted parameters
are not a global minimiser of the empirical logistic loss within this architecture.
This run illustrates an optimisation difficulty as well as noisy data; it does
not by itself establish that gradient descent has reached a local minimum.

```python
centre_X = t.tensor([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
centre_y = t.tensor([[0.0], [0.0], [1.0], [1.0]])

def lecture_score(inputs):
    difference = inputs[:, 0:1] - inputs[:, 1:2]
    return t.relu(difference) + t.relu(-difference) - 0.5

centre_scores = lecture_score(centre_X)
centre_mistakes = int(((centre_scores > 0).to(centre_y.dtype) != centre_y).sum().item())
noisy_scores = lecture_score(X)
print('Mistakes on the four centres:', centre_mistakes)
print('Mistakes on the noisy observations:', mistakes_from_scores(noisy_scores))
print('Empirical logistic loss on the noisy observations:', loss_from_scores(noisy_scores).item())
```

</details>

In [ ]:
# Supplied: compare your fitted network with the network constructed in the lecture.
plot_lecture_comparison(narrow_grid_scores, narrow_loss_history[-1],
                        narrow_error_history[-1], lecture_score)

### Why did gradient descent miss these parameters?

The lecture's network has both a lower empirical logistic loss and fewer mistakes on the same noisy data. Noise alone therefore cannot explain our poor fitted result. **Have we become stuck near a suboptimal local minimum?**

The neural-network empirical logistic loss is not convex in its parameters. Gradient descent uses derivatives at its current parameters; it does not search every possible setting. It can make very little progress near a local minimum or in a flat region. Our comparison proves that the original fit is not a global minimum.

Let us try the lecture's first heuristic: **start again from different random initialisations and keep the best fitted result**. We will then try its other heuristic by changing the architecture.

### 5.2. Try five initialisations and keep the best run

We now test the lecture's heuristic of **trying different initialisations**.
Use the fixed list **`[0, 1, 2, 3, 4]`**, declared before fitting. Run every seed
and keep every result, including any disappointing fits. The seeds make the
comparison reproducible; a new starting point does not guarantee improvement.
The original seed was chosen to illustrate a poor fit. Here we will compare a
complete set of five restarts rather than report only a successful attempt.

**Exercise 9 — fit five runs and select by final empirical logistic loss.**

**Part A: repeat your calculations.** The supplied outer loop visits each `seed`.
Within that loop:

1. After `t.manual_seed(seed)`, reuse only the four `t.randn(...)` assignments
   from Exercise 1, in the order `b1`, `W1`, `b2`, `W2`. Create **new tensors**
   each time, with the same two-neuron shapes and `requires_grad=True`.
2. Copy your entire completed fitting cell from Exercise 5 below those assignments,
   indented one extra level so it runs inside the seed loop. Keep learning rate
   `0.001` and 1,000 updates. Its new history lists and `boundary_scores` dictionary
   must also be created inside the seed loop, before that run's updates.
3. Keep the supplied recording lines after the fitting loop, still inside the
   seed loop. They save that run's histories, final boundary and four parameters.

Use the existing `net`, empirical logistic loss and mistake functions. **Do not regenerate `X` or `y`,
or copy Exercise 1's fixed `t.manual_seed(...)` line into this loop.** Only the
initial parameters change. Rerunning this whole cell repeats all five runs.

The supplied dictionaries store each result under its seed, as the earlier
boundary dictionary stored pictures under update numbers. For example,
`restart_loss_histories[seed][-1]` is that run's final empirical logistic loss. Each run creates new
parameter tensors and new histories, so recording them keeps the previous runs
available when the next run reuses the same variable names.

In [ ]:
# Supplied: fix the five seeds and prepare separate records for every run.
restart_seeds = [0, 1, 2, 3, 4]
restart_loss_histories = {}
restart_error_histories = {}
restart_grid_scores_by_seed = {}
restart_parameters = {}
input_features = 2
hidden_features = 2
output_features = 1

for seed in restart_seeds:
    t.manual_seed(seed)

    # YOUR CODE: create the four new parameter tensors, as in Exercise 1.

    # YOUR CODE: insert your completed Exercise 5 fitting code, indented here.

    # Supplied: keep this completed run before drawing the next initialisation.
    restart_loss_histories[seed] = loss_history
    restart_error_histories[seed] = error_history
    restart_grid_scores_by_seed[seed] = boundary_scores[num_updates]
    restart_parameters[seed] = [b1, W1, b2, W2]

<details>
<summary>Show answer — repeat the same fitting procedure for all five seeds</summary>

The outer loop changes only the random starting parameters. Each run has fresh
parameters and recording lists, and uses the same data and 1,000-update budget.
The network formula and gradient-descent update are unchanged.

```python
# Supplied: fix the five seeds and prepare separate records for every run.
restart_seeds = [0, 1, 2, 3, 4]
restart_loss_histories = {}
restart_error_histories = {}
restart_grid_scores_by_seed = {}
restart_parameters = {}
input_features = 2
hidden_features = 2
output_features = 1

for seed in restart_seeds:
    t.manual_seed(seed)
    b1 = t.randn(hidden_features, requires_grad=True)
    W1 = t.randn(input_features, hidden_features, requires_grad=True)
    b2 = t.randn(output_features, requires_grad=True)
    W2 = t.randn(hidden_features, output_features, requires_grad=True)

    learning_rate = 0.001
    num_updates = 1000
    boundary_scores = {}

    scores = net(X)
    loss_history = [loss_from_scores(scores).item()]
    error_history = [mistakes_from_scores(scores)]

    for step in range(1, num_updates + 1):
        for parameter in (b1, W1, b2, W2):
            parameter.grad = None
        L = loss_from_scores(net(X))
        L.backward()

        with t.no_grad():
            b1.add_(-learning_rate * b1.grad)
            W1.add_(-learning_rate * W1.grad)
            b2.add_(-learning_rate * b2.grad)
            W2.add_(-learning_rate * W2.grad)

        scores = net(X)
        loss_history.append(loss_from_scores(scores).item())
        error_history.append(mistakes_from_scores(scores))

        # Supplied: record the boundary at these completed updates.
        if step in (1, 100, num_updates):
            boundary_scores[step] = save_boundary(net)

    # Supplied: keep this completed run before drawing the next initialisation.
    restart_loss_histories[seed] = loss_history
    restart_error_histories[seed] = error_history
    restart_grid_scores_by_seed[seed] = boundary_scores[num_updates]
    restart_parameters[seed] = [b1, W1, b2, W2]
```

</details>

**Part B: keep the run with the lowest final empirical logistic loss.**
Use the same objective that gradient descent minimises, rather than choosing by
mistake count or by how attractive a boundary looks.

Start with the first seed as `best_seed`. The supplied loop visits the remaining
seeds (`restart_seeds[1:]` skips the first entry). For each one, use an `if`
statement to compare `restart_loss_histories[seed][-1]` with
`restart_loss_histories[best_seed][-1]`. If the new empirical logistic loss is smaller, assign
`best_seed = seed`. With a strict `<` comparison, a tie keeps the earlier seed.

Print the selected seed, its final empirical logistic loss and its mistake count. Then run the
supplied saving cell: it retrieves the four tensors stored for that seed, so
`net` uses the selected model rather than whichever seed happened to run last.
The names beginning with `restart_` in later comparisons refer to this best run.

In [ ]:
best_seed = restart_seeds[0]
for seed in restart_seeds[1:]:
    # YOUR CODE: compare final empirical logistic losses and update best_seed when appropriate.

# YOUR CODE: print the selected seed, its final empirical logistic loss and its mistake count.

<details>
<summary>Show answer — select the run using the fitting objective</summary>

The comparison keeps the smallest final empirical logistic loss seen so far. With these settings,
the selected seed is **3**, with empirical logistic loss about **30.173**
and **12 mistakes**. It is the best of these five completed
runs according to the empirical logistic loss, not a claim of a global minimum.

```python
best_seed = restart_seeds[0]
for seed in restart_seeds[1:]:
    if restart_loss_histories[seed][-1] < restart_loss_histories[best_seed][-1]:
        best_seed = seed

print('Selected seed:', best_seed)
print('Final empirical logistic loss:', restart_loss_histories[best_seed][-1])
print('Mistakes:', restart_error_histories[best_seed][-1])
```

</details>

In [ ]:
# Supplied: keep the selected model, not merely its reported empirical logistic loss.
best_parameters = restart_parameters[best_seed]
b1 = best_parameters[0]
W1 = best_parameters[1]
b2 = best_parameters[2]
W2 = best_parameters[3]
restart_loss_history = restart_loss_histories[best_seed]
restart_error_history = restart_error_histories[best_seed]
restart_grid_scores = restart_grid_scores_by_seed[best_seed]

print('Seed   Final empirical logistic loss   Mistakes')
for seed in restart_seeds:
    print(seed, restart_loss_histories[seed][-1], restart_error_histories[seed][-1])
print('Selected seed:', best_seed)

**Part C: compare every run.** Run the supplied figures. They show the original
fit and all five restarts, with the selected run labelled. We keep all results
visible even though we retain only one model for subsequent comparisons.

1. How much do final empirical logistic losses, mistake counts and boundaries vary across seeds?
   How do the five runs compare with the original fit?
2. Which run does our selection rule keep? Compare it with the lecture's
   constructed score too: must a lower empirical logistic loss mean fewer mistakes?
3. Why must every restart create new parameters? Would continuing to update
   the last run implement the same strategy?
4. Does choosing the best of five guarantee a global minimum or good predictions
   on new data? How many parameter updates did the five attempts require in total?

Write down your answers before opening the explanation.

In [ ]:
# Supplied: show the original run and all five restarts, keeping the architecture fixed.
plot_restart_curves(restart_seeds, restart_loss_histories, restart_error_histories,
                    narrow_loss_history, narrow_error_history, best_seed)

In [ ]:
# Supplied: show every final boundary; identify the restart selected by final empirical logistic loss.
plot_restart_boundaries(
    restart_seeds, restart_grid_scores_by_seed, restart_loss_histories, restart_error_histories,
    narrow_grid_scores, narrow_loss_history[-1], narrow_error_history[-1], best_seed)

<details>
<summary>Show answer — comparing initialisations</summary>

**The starting point matters.** All five restarts improve on the original empirical logistic loss
69.308 and its 47 mistakes, but their final results differ.
Our rule selects seed **3**, with empirical logistic loss **30.173** and
**12 mistakes**. Every run has the same nine-parameter architecture,
data, empirical logistic loss, learning rate and 1,000-update budget. No starting weights were set
to the lecture's construction: each run draws them with `t.randn`.

The selected run has lower empirical logistic loss than the lecture's score (about 54.071),
yet makes more mistakes (12 versus 8). Empirical logistic loss uses the confidence of each prediction;
mistakes only use which side of zero each score lies on. We select by empirical logistic loss because
it is our fitting objective. Continuing an existing run would explore further
from its current parameters, rather than try a different starting point.

Five attempts cost **5,000 updates in total**, in addition to the original fit.
Keeping the best of them is a practical search strategy, not a guarantee that
another initialisation will improve the fit. It neither proves convergence to
local minima nor establishes a global minimum or good predictions on unseen data.
</details>

## 6. What changes with 100 hidden neurons?

We now test **changing the architecture**. Increase the hidden width from
$d_1=2$ to $d_1=100$, keeping the same 100 observations and one scalar output.
The figures compare this fit with both the original two-neuron run and the best
of our five restarts. Every individual run has 1,000 updates; finding the best
restart required five such runs.

**Exercise 10 — widen and fit your network.**

**Part A.** Copy your initialisation from Exercise 1 into the new cell below and
set `hidden_features = 100`. Use the supplied seed 94.
Predict the shapes and total parameter count first. Which tensor keeps its shape?
Changing the dimensions also changes the initial parameter arrays.
Keep the earlier exercise cells unchanged.

In [ ]:
t.manual_seed(94)  # Supplied: repeatable starting parameters.
input_features = 2
hidden_features = 100
output_features = 1

# YOUR CODE: create b1, W1, b2 and W2; print shapes and count scalar parameters.

<details>
<summary>Show answer — initialising the wider network</summary>

The shapes are `W1`: `(2, 100)`, `b1`: `(100,)`, `W2`: `(100, 1)`, and `b2`: `(1,)`.
There are 401 scalar parameters. Only `b2` keeps its shape; the network formula
still applies with these dimensions.

```python
t.manual_seed(94)  # Supplied: repeatable starting parameters.
input_features = 2
hidden_features = 100
output_features = 1

b1 = t.randn(hidden_features, requires_grad=True)
W1 = t.randn(input_features, hidden_features, requires_grad=True)
b2 = t.randn(output_features, requires_grad=True)
W2 = t.randn(hidden_features, output_features, requires_grad=True)
print('W1:', W1.shape, 'b1:', b1.shape)
print('W2:', W2.shape, 'b2:', b2.shape)
print('Scalar parameters:', W1.numel() + b1.numel() + W2.numel() + b2.numel())
```

</details>

**Part B.** After Part A, copy your completed fitting cell from Exercise 5 into
the cell below and run it. Keep learning rate `0.001` and 1,000 updates.
Explain why the same network function and update code work with wider tensors.
To repeat from the initial parameters, rerun Part A before fitting again.

In [ ]:
# YOUR CODE: reuse your completed fitting cell from Exercise 5, after running Part A.

<details>
<summary>Show answer — the same update rule with more neurons</summary>

Use the complete fitting code from **Exercise 5** unchanged, after the 100-neuron
initialisation. The matrix and gradient shapes follow the parameter shapes.
The learning rate, update count and mathematical update rule stay the same.

</details>

Run the supplied cell to keep this experiment's results under names beginning
with `wide_`, then compare the fits using the supplied figures.

In [ ]:
# Supplied: keep the 100-neuron results for comparison.
assert W1.shape == (2, 100)
assert len(loss_history) == len(error_history) == 1001
wide_loss_history = loss_history
wide_error_history = error_history
wide_grid_scores = boundary_scores[num_updates]

**Exercise 11 — compare the two widths.**

**Part A: curves.** Run the supplied cell to compare the original two-neuron
fit, the best of its five restarts, and the 100-neuron fit. The saved histories
begin with `narrow_`, `restart_` and `wide_`, respectively. Compare both empirical logistic losses and mistake counts.

In [ ]:
# Supplied: compare both two-neuron fits with the wider network.
plot_network_curves(
    ['2 neurons (original)', f'2 neurons (best restart: seed {best_seed})', '100 hidden neurons'],
    [narrow_loss_history, restart_loss_history, wide_loss_history],
    [narrow_error_history, restart_error_history, wide_error_history])

**Part B: boundaries.** Run the supplied cell to compare the affine classifier,
the original two-neuron fit, its best restart, and the wider network. Each panel
uses saved grid scores, so no earlier fit needs to be rerun.

Which has the smaller empirical logistic loss and fewer mistakes? How do their boundaries differ?
Does widening guarantee a better fit for every initialisation, or better
predictions on unseen observations?

In [ ]:
# Supplied: compare the final affine and neural-network boundaries.
plot_boundaries(
    [affine_grid_scores, narrow_grid_scores, restart_grid_scores, wide_grid_scores],
    ['Affine', '2 neurons (original)', f'2 neurons (best restart: seed {best_seed})', '100 hidden neurons'],
    [affine_error_history[-1], narrow_error_history[-1], restart_error_history[-1], wide_error_history[-1]])

<details>
<summary>Show answer — comparing widths</summary>

The original two-neuron fit has empirical logistic loss 69.308 and 47 mistakes;
the 100-neuron network reaches 7.159 with 2 mistakes and
separates most observations in this run. Wider tensors also have different
initial parameters. This experiment does not guarantee better fitting for every
initialisation, a global minimum, or better predictions on unseen data.
The best of the five two-neuron restarts already reaches empirical logistic loss 30.173
and 12 mistakes without changing the architecture. Selecting it
required five runs, so this is not a comparison at equal total computing effort.
</details>

### Final comparison

The table brings together the original two-neuron fit, the best of its five
restarts, and the 100-neuron fit. All use the same observations and 1,000 updates
per run; selecting the best restart required five runs. The parameter counts
are those you derived in Exercises 1 and 10.

In [ ]:
# Supplied: collect your final measurements for interpretation.
print('Hidden neurons                 Parameters   Final empirical logistic loss   Mistakes')
print(f'2, original initialisation              9   {narrow_loss_history[-1]:10.3f}   {narrow_error_history[-1]}')
print(f'2, best of 5 (seed {best_seed})               9   {restart_loss_history[-1]:10.3f}   {restart_error_history[-1]}')
print(f'100                                   401   {wide_loss_history[-1]:10.3f}   {wide_error_history[-1]}')

You have translated a network architecture into tensors, implemented its score
function, calculated empirical logistic losses and mistakes, obtained derivatives,
and used them in your own gradient-descent loop. The supplied plots let you
compare your fitted networks with the affine classifier on the same data.

You also evaluated the lecture's XOR network on noisy observations, tried five
random initialisations and kept the run with the lowest final empirical logistic loss, then increased the hidden width from 2 to 100 neurons. These experiments
illustrate two of the lecture's heuristic strategies: trying different
initialisations and widening the network. We kept the dataset, empirical logistic loss, learning rate and update count per run fixed. Their effects
on fitting must be checked experimentally rather than assumed.